## Módulo 2 - Aprendizagem Automática para Mobilidade e Qualidade do Ar

### 1. Introdução

**Contexto**

A câmara municipal pretende prever a qualidade do ar para ajustar planos de tráfego e alertas à população.

**Tarefas**
1. Fazer EDA (estatísticas, histogramas, correlações, missing values)
2. Criar pré-processamento (normalização, imputação, split treino/teste)
3. Treinar ≥ 2 algoritmos de classificação (ex.: Logistic Regression, Random Forest) para prever `air_quality_good`
4. Treinar pelo menos um algoritmo de regressão para prever NO2
5. Comparar métricas e justificar a escolha do modelo final
6. Guardar modelos e métricas em ficheiros (`.pkl`, `.csv`)

### 2. Dados

Trabalhamos sobre o `data/clean_air_quality.csv` (versão tratada produzida no Módulo 1), em vez do ficheiro raw original. Vantagens:

- Dados já filtrados a Lisboa+Porto (1442 linhas)
- Colunas vazias já removidas
- Permite comparação directa com a Rede Bayesiana do Módulo 1 no final

**Ponto de atenção:** a coluna `air_quality_good` foi **recalculada no Módulo 1** com a seguinte fórmula:

`air_quality_good = False  quando  (NO2 >= 30 µg/m³) AND (humidade >= 80%)`

Razão: a coluna original era sempre `True` em Lisboa+Porto, sem variação para treinar modelos. A nova fórmula introduz variabilidade (~10% de casos "má"). Como NO2 e humidade definem a target, vão ter que ser excluídas das features na classificação (caso contrário data leakage).

### 3. Setup e carregamento

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ----------------------------------------------------------
# Paths — alterar aqui se mudarmos a localização dos ficheiros
# ----------------------------------------------------------
DATA_CSV    = '../data/clean_air_quality.csv'   # input (vem do Módulo 1)
METRICS_CSV = 'metrics.csv'                     # output partilhado pelos .py
MODEL_DIR   = '.'                               # onde guardar os .pkl

**Targets e features a excluir (porquê)**

Cada uma das tarefas (classificação e regressão) tem uma target diferente. E como as features estão relacionadas entre si, há sempre o risco de **data leakage** — passar ao modelo uma feature que está demasiado próxima da target, fazendo o modelo "copiar" em vez de aprender.

No nosso caso:

- **Classificação** prevê `air_quality_good`, que foi definida no Módulo 1 como `(NO2 >= 30) AND (humidade >= 80)`. Logo NO2 e humidade definem a target → têm que ser excluídas das features (caso contrário 100% accuracy ilusório).
- **Regressão** prevê NO2. Como `air_quality_good` foi derivada de NO2, também tem que ser excluída das features da regressão.

A célula seguinte regista isto em constantes (`TARGET_*` e `LEAK_*`) que vão ser reutilizadas nos `.py` de treino.

In [ ]:
# Targets de cada tarefa
TARGET_CLASSIFICATION = 'air_quality_good'   # boolean: ar bom (True) / mau (False)
TARGET_REGRESSION     = 'NO2'                # numérico: concentração em µg/m³

# Features a excluir por leakage
LEAK_CLASSIFICATION = ['NO2', 'humidity_percent']   # definem air_quality_good
LEAK_REGRESSION     = ['air_quality_good']          # deriva de NO2

In [ ]:
# Carregar dataset
df = pd.read_csv(DATA_CSV, sep=';')

print(f"Dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Cidades : {df['city'].unique().tolist()}")
print(f"Período : meses {sorted(df['month'].unique().tolist())}")
print(f"\nPrimeiras linhas:")
df.head()